Consertando encoding

In [ ]:
file = "/content/drive/MyDrive/NLP2/ep2-train.csv"

In [ ]:
!pip install chardet
import chardet

with open(file, "rb") as f:
    enc = chardet.detect(f.read(100000))
print(enc)

In [ ]:
with open(file, "r", encoding="cp1252", errors="replace") as src:
    text = src.read()

with open(file, "w", encoding="utf-8", newline="\n") as dst:
    dst.write(text)

Codigo do problema

In [20]:
import polars as pl
import numpy as np
from sklearn.model_selection import GridSearchCV, StratifiedKFold, cross_val_score, cross_validate
from sklearn.pipeline import Pipeline, FeatureUnion
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.feature_selection import SelectKBest, chi2
from sklearn.svm import LinearSVC
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import VotingClassifier, RandomForestClassifier
from xgboost import XGBClassifier

In [2]:
file = "/content/drive/MyDrive/NLP2/ep2-train.csv"

In [4]:
df = pl.read_csv(source=file, separator=";").filter(pl.col("req_text").is_not_null()).unique(subset=["req_text", "profession"])
print(df)

shape: (36_780, 2)
┌─────────────────────────────────┬────────────┐
│ req_text                        ┆ profession │
│ ---                             ┆ ---        │
│ str                             ┆ str        │
╞═════════════════════════════════╪════════════╡
│ Prezados, fiz o saque do meu F… ┆ government │
│ Em decorrência da denegação do… ┆ government │
│ Prezados, bom dia! No intuito … ┆ government │
│ Boa noite! Gostaria de saber s… ┆ academic   │
│ Gostaria de pedir autorização … ┆ government │
│ …                               ┆ …          │
│ Prezados, Boa Tarde! Esta soli… ┆ government │
│ Com base no Art. 37 da Constit… ┆ government │
│ Gostaria de saber se existe có… ┆ academic   │
│ Tendo em vista que o requerent… ┆ government │
│ Bom dia Solicito o Projeto Pol… ┆ academic   │
└─────────────────────────────────┴────────────┘


Modelo 1: ~74%

In [ ]:
pipeline = Pipeline([
    ("features", FeatureUnion([
        ("word", TfidfVectorizer(analyzer="word", ngram_range=(1,2), min_df=3)),
        ("char", TfidfVectorizer(analyzer="char", ngram_range=(1,5), min_df=3))
    ])),
    ("kbest", SelectKBest(chi2, k=22000)),
    ("clf", LinearSVC(C=0.9))
])


Modelo 2: ~75%

In [10]:
pipeline = Pipeline([
    ("features", FeatureUnion([
        ("word", TfidfVectorizer(
            analyzer="word",
            ngram_range=(1,3),
            min_df=2,
            max_df=0.85,
            sublinear_tf=True
        )),
        ("char", TfidfVectorizer(
            analyzer="char",
            ngram_range=(3,6),
            min_df=2,
            max_features=30000
        ))
    ])),
    ("kbest", SelectKBest(chi2, k=32000)),
    ("clf", LogisticRegression(
        C=1.5,
        max_iter=1000,
        class_weight='balanced',
        random_state=42
    ))
])

Modelo 3: 77.8%

In [28]:
pipeline = Pipeline([
    ("tfidf", TfidfVectorizer(
        analyzer="word",
        ngram_range=(1, 3),
        min_df=2,
        max_df=0.85,
        sublinear_tf=True
    )),
    ("kbest", SelectKBest(chi2, k=18000)),
    ("clf", VotingClassifier(
        estimators=[
            ('lr', LogisticRegression(C=2.0, max_iter=1500, class_weight='balanced')),
            ('rf', RandomForestClassifier(n_estimators=200, max_depth=20, min_samples_split=5, class_weight='balanced', random_state=42)),
            ('xgb', XGBClassifier(n_estimators=150, max_depth=6, learning_rate=0.1, random_state=42, eval_metric='mlogloss'))
        ],
        voting='soft',
        weights=[2, 1, 2]
    ))
])

In [ ]:
type_metrics = [
    'accuracy',
    'precision',
    'recall',
    'f1',
    'f1_macro'
]
score_train = []
for file in [file]:
  csv = (
        pl.read_csv(source=file, separator=";")
        .filter(pl.col("req_text").is_not_null())
        .unique(subset=["req_text", "profession"])
    )

  label_text = list(set(csv["profession"]))
  labels = [label_text.index(style) for style in csv["profession"]]
  texts = [text for text in csv['req_text']]

  scores = cross_validate(pipeline, texts, labels, cv=10, scoring=type_metrics)
  score_train.append(scores)


In [ ]:
print(np.mean(score_train[0]['test_accuracy']))

Salvando o modelo

In [25]:
csv = (
        pl.read_csv(source=file, separator=";")
        .filter(pl.col("req_text").is_not_null())
        .unique(subset=["req_text", "profession"])
    )

In [26]:
print(csv)

shape: (36_780, 2)
┌─────────────────────────────────┬────────────┐
│ req_text                        ┆ profession │
│ ---                             ┆ ---        │
│ str                             ┆ str        │
╞═════════════════════════════════╪════════════╡
│ Solicito informação de Assiste… ┆ government │
│ Correção da Portaria MEC nº 1.… ┆ government │
│ Requerimento de Login e Senha,… ┆ academic   │
│ Prezada(o), bom dia. Gostaria,… ┆ private    │
│ Prezados, boa tarde. De forma … ┆ government │
│ …                               ┆ …          │
│ Bom dia prezados colegas. Tive… ┆ government │
│ Sobre a OBRA DE REFORMA E AMPL… ┆ academic   │
│ Gostaria de saber quanto a Uni… ┆ government │
│ Prezados senhores, Por motivo … ┆ academic   │
│ Prezados, estou tentando fazer… ┆ academic   │
└─────────────────────────────────┴────────────┘


In [27]:
import joblib

label_text = list(set(csv["profession"]))
labels = [0 if style == label_text[0] else 1 for style in csv["profession"]]
texts = [text for text in csv['req_text']]

pipeline.fit(texts, labels)

joblib.dump(pipeline, 'text_classifier_model.pkl')

loaded_pipeline = joblib.load('text_classifier_model.pkl')

predictions = loaded_pipeline.predict(["new text to classify"])
probabilities = loaded_pipeline.predict_proba(["new text to classify"])

KeyboardInterrupt: 